# Notebook 06 — Walk-Forward Validation and Model Evaluation

**Synopsis objective 5:** evaluate and compare model performance against baselines using RMSE / MAE / WMAPE / MASE under walk-forward (time-respecting) cross-validation.

This is the decision notebook of the whole pipeline. Notebooks 01–05 cleaned the raw Rossmann extract, engineered calendar/lag/rolling features, and established two zero-training baselines (seasonal-naive, moving-average). Here we:

1. Build 5 expanding-window, date-ordered folds that never let a model see the future.
2. Fit the two baselines, a Ridge regression, and a LightGBM regressor identically on every fold.
3. Score every model with the same four metrics (RMSE, MAE, WMAPE, MASE) on the Sales scale.
4. Decide — using the real numbers produced below, not intuition — whether LightGBM's extra complexity is actually earning its keep over Ridge and over the baselines.
5. Persist a final LightGBM model, refit on all trainable history, for notebook 08's test-set forecast.

Everything downstream (the model-choice discussion and the forecast notebook) depends on the comparison table produced here, so every step is kept simple, auditable, and leakage-checked.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

## Why one global model, not 1,115 per-store models

The Rossmann dataset covers 1,115 stores. A tempting alternative to the global model built below would be to fit a separate model *per store* — 1,115 independent small time-series models. We deliberately reject that design:

- **Each store only has ~900 rows of history to begin with**, and that shrinks further once walk-forward validation folds are held out. That is already too little data for 1,115 independent per-store models to learn anything beyond the day-of-week pattern — and barely enough to fit a stable per-store Ridge regression, let alone a per-store gradient-boosted tree model.
- **1,115x the engineering and operational overhead** — 1,115 models to tune, validate, serialize, monitor for drift, and re-train on every refresh cycle, for no demonstrated accuracy gain over a single model that already sees `Store` as a feature.
- **A global model with `Store` (and store attributes such as `StoreType`, `Assortment`, `CompetitionDistance`) as features learns the shared seasonal and promotional structure that is common to all stores** — the weekly cycle, the state-holiday effect, the promo lift — while the `Store` categorical and the per-store numeric attributes still let it differentiate individual store behavior. It gets the best of both worlds: pooled statistical power *and* store-level discrimination.
- This is also how the strongest public solutions to the real Rossmann Kaggle competition were structured: a single gradient-boosted model trained across all stores with `Store` as a categorical feature, rather than thousands of independent per-store models.

The data loaded below is a sampled subset of 15 stores (kept small so this course project's notebooks run quickly), but the same argument holds at that scale too — splitting 15 stores into 15 separate models would leave each one with only a few hundred training rows once validation folds are held out, the same data-starvation problem, just smaller in absolute numbers. The design choice is scale-invariant, which is exactly why it is also how the full 1,115-store competition was won.

In [ ]:
import numpy as np
import pandas as pd
import joblib
import lightgbm as lgb
from pathlib import Path

from src.data_io import load_parquet
from src.features import NUMERIC_FEATURE_COLS, CATEGORICAL_FEATURE_COLS
from src.models import (
    seasonal_naive_predict, moving_average_predict,
    build_ridge_pipeline, build_lightgbm_model, set_categorical_dtypes,
)
from src.validation import walk_forward_splits
from src.metrics import evaluate_all

## The walk-forward validation scheme

Random k-fold cross-validation would be invalid here: shuffling rows lets a model trained on a shuffled fold see sales from *after* the date it is being asked to predict, silently leaking future information into the past. `walk_forward_splits` avoids this entirely:

- It builds **5 folds**, each an **expanding window**: fold 1 trains on the earliest slice of history, fold 5 trains on (almost) everything up to that fold's cut-off date. Every fold's training set is a strict superset of the previous fold's.
- Each **validation block is 42 days (6 weeks) long**, non-overlapping, and immediately follows its fold's training cut-off. 42 days is deliberately chosen to match the horizon length of the real Kaggle Rossmann `test.csv` set, so the validation error measured here is representative of the actual forecast horizon the model will eventually be judged on.
- **No shuffling happens anywhere** in this notebook. `walk_forward_splits` builds `train_mask`/`val_mask` purely from boolean comparisons on `Date` — there is no `sample`, no `shuffle=True`, no random split of any kind. The loop below adds an explicit guard, `assert train_df["Date"].max() < val_df["Date"].min()`, which independently re-proves for *every single fold* that the training dates end strictly before the validation dates begin. If that assertion ever fired, it would mean the fold construction let future information bleed into training — it never does.

Printing the fold boundaries below makes the expanding-window structure concrete: `train_end` keeps moving forward fold over fold, and the five validation windows tile the most recent ~7 months of history back-to-back with no gaps and no overlap.

In [ ]:
df = load_parquet("../data/processed/train_features.parquet").sort_values(["Store", "Date"]).reset_index(drop=True)
FEATURE_COLS = NUMERIC_FEATURE_COLS + CATEGORICAL_FEATURE_COLS
splits = walk_forward_splits(df["Date"], n_folds=5, val_days=42)
print(f"{len(splits)} folds")
for s in splits:
    print(s["fold"], "train<=", s["train_end"].date(), "val", s["val_start"].date(), "->", s["val_end"].date())

## What happens inside each fold

For every one of the 5 folds, the loop below fits four models on that fold's training slice and scores all four on that *same* fold's validation slice, so the comparison is fair — no model gets an easier or harder validation set than any other:

- **`seasonal_naive`** and **`moving_average`** are zero-training baselines — they simply read the already-computed `lag_7` and `roll_mean_7` columns, i.e. "sales will repeat last week" and "sales will equal the trailing 7-day average."
- **Ridge regression** is fit on the engineered feature matrix with one-hot-encoded categoricals and standardized numerics (`build_ridge_pipeline`), trained on `log_sales` (log1p of Sales) for variance stabilization.
- **LightGBM** is fit on the same feature matrix, but with native categorical support: `set_categorical_dtypes` casts the categorical columns to pandas `category` dtype so LightGBM splits on them directly rather than through one-hot columns. It is also trained on `log_sales`, with early stopping monitored against that fold's own validation set so training stops once validation loss stops improving.

Every prediction is produced on the log scale and immediately inverse-transformed with `np.expm1(...)` before it touches a metric — RMSE, MAE, WMAPE, and MASE are all computed on real Sales (daily revenue in whatever currency the store reports), never on the log scale, because that is the unit the business actually cares about.

One subtlety worth naming honestly: because LightGBM's early stopping also watches this same fold's validation set, it gets to choose its iteration count using validation information, a mild structural advantage Ridge does not get. Keep that in mind when reading the results below — it means any case where Ridge still comes out ahead is, if anything, an *understatement* of how much better Ridge is doing.

## Why `evaluate_all(..., train_groups=train_df["Store"])` is not optional

MASE scales the model's mean absolute error by the mean absolute error of a seasonal-naive forecast computed on the *training* data (see `src/metrics.py`). For a single store's series, that scale is just `mean(|y[t] - y[t-7]|)` over the training rows.

But `train_df` here is **not** a single store's series — it is `Store`/`Date`-sorted rows from a global model's training set, i.e. a concatenation of (in this project's sampled data) 15 different stores' time series stacked on top of each other, or 1,115 of them at full Kaggle scale. If `mase()` were called without `train_groups`, the lag-7 difference `y_train[t] - y_train[t-7]` would be computed *positionally* across the entire concatenated array — meaning for the handful of rows sitting at the boundary between, say, store 3's last week and store 4's first week, the "naive error" would be the absolute difference between two completely unrelated stores' sales levels. That contamination would silently bias the MASE denominator on every fold, in a way that is easy to miss because the code still runs without error — it would just quietly report the wrong number.

Passing `train_groups=train_df["Store"]` tells `mase()` to compute the lag-7 naive-error scale **within each store separately** and only pool the per-store error arrays afterward (see the `for group in np.unique(train_groups)` loop inside `mase()`). This is the single easiest line in the whole evaluation loop to get silently wrong, and it is non-negotiable for any multi-entity global model, not just this one.

In [ ]:
results = []
last_fold_val_df = None
last_fold_lgbm_pred = None

for split in splits:
    train_df = df.loc[split["train_mask"]]
    val_df = df.loc[split["val_mask"]]
    assert train_df["Date"].max() < val_df["Date"].min(), "fold-boundary leakage check failed"

    row = {"fold": split["fold"], "n_train": len(train_df), "n_val": len(val_df)}

    for name, pred_fn in [("seasonal_naive", seasonal_naive_predict), ("moving_average", moving_average_predict)]:
        pred = pred_fn(val_df)
        m = evaluate_all(val_df["Sales"], pred, train_df["Sales"], seasonal_period=7, train_groups=train_df["Store"])
        row.update({f"{name}_{k}": v for k, v in m.items()})

    ridge = build_ridge_pipeline(NUMERIC_FEATURE_COLS, CATEGORICAL_FEATURE_COLS)
    ridge.fit(train_df[FEATURE_COLS], train_df["log_sales"])
    ridge_pred = np.expm1(ridge.predict(val_df[FEATURE_COLS]))
    m = evaluate_all(val_df["Sales"], ridge_pred, train_df["Sales"], seasonal_period=7, train_groups=train_df["Store"])
    row.update({f"ridge_{k}": v for k, v in m.items()})

    train_cat = set_categorical_dtypes(train_df, CATEGORICAL_FEATURE_COLS)
    val_cat = set_categorical_dtypes(val_df, CATEGORICAL_FEATURE_COLS)
    lgbm = build_lightgbm_model()
    lgbm.fit(
        train_cat[FEATURE_COLS], train_cat["log_sales"],
        eval_set=[(val_cat[FEATURE_COLS], val_cat["log_sales"])],
        callbacks=[lgb.early_stopping(50, verbose=False)],
    )
    lgbm_pred = np.expm1(lgbm.predict(val_cat[FEATURE_COLS]))
    m = evaluate_all(val_df["Sales"], lgbm_pred, train_df["Sales"], seasonal_period=7, train_groups=train_df["Store"])
    row.update({f"lightgbm_{k}": v for k, v in m.items()})

    results.append(row)
    last_fold_val_df = val_df
    last_fold_lgbm_pred = lgbm_pred

In [ ]:
results_df = pd.DataFrame(results)
results_df

In [ ]:
summary = results_df.drop(columns=["fold", "n_train", "n_val"]).agg(["mean", "std"]).T
summary

In [ ]:
wmape_cols = ["seasonal_naive_wmape", "moving_average_wmape", "ridge_wmape", "lightgbm_wmape"]
wins = (results_df["lightgbm_wmape"] == results_df[wmape_cols].min(axis=1)).sum()
print(f"LightGBM had the lowest WMAPE in {wins}/{len(results_df)} folds")
print(f"Mean LightGBM MASE across folds: {results_df['lightgbm_mase'].mean():.3f}")

## Reading the comparison table: does LightGBM actually win?

Averaged across the 5 folds (the `mean` column of the `summary` table above):

| Model | RMSE | MAE | WMAPE | MASE |
|---|---|---|---|---|
| Seasonal-naive (lag-7) | ≈1399 | ≈864 | ≈13.3% | ≈0.73 |
| Moving-average (roll_mean_7) | ≈1711 | ≈1257 | ≈19.4% | ≈1.06 |
| Ridge | ≈336 | ≈260 | ≈4.00% | ≈0.22 |
| LightGBM | ≈368 | ≈286 | ≈4.40% | ≈0.24 |

Two things stand out:

1. **Both learned models crush both naive baselines**, by a wide margin, on every metric and in every fold — WMAPE drops from ≈13–19% for the baselines to ≈4% for Ridge/LightGBM, and MASE drops from ≈0.73–1.06 down to ≈0.22–0.24. A MASE around 1.0 means "about as good as copying last week's sales"; the moving-average baseline barely clears that bar (mean MASE ≈1.06, i.e. slightly *worse* than naive repetition), while Ridge and LightGBM are both roughly 4–5x more accurate than the seasonal-naive reference. The feature engineering and modeling effort in this project is clearly earning its keep relative to "do nothing clever."
2. **LightGBM does not beat Ridge here.** The diagnostic printed above shows LightGBM had the lowest WMAPE in **0 of 5 folds** — Ridge's WMAPE is lower than LightGBM's in every single fold (e.g. fold 1: 4.00% vs 4.65%; fold 5: 3.92% vs 4.36%), and the same ordering holds for MASE (Ridge's mean MASE ≈0.218 vs LightGBM's ≈0.240). On this dataset, the extra flexibility of a 1000-tree, 63-leaf gradient-boosting model is not translating into lower validation error than a linear Ridge regression on the same one-hot/standardized features — and recall from the note two cells up that LightGBM's early stopping already had a structural advantage (it alone got to peek at validation loss to pick its iteration count), which makes Ridge's edge here more notable, not less.

A plausible explanation: this course project works with a sampled subset of 15 stores (≈9,000–11,000 training rows per fold), and `build_lightgbm_model()`'s defaults (1000 estimators, 63 leaves) carry a lot of capacity for that amount of data even with 50-round early stopping. Ridge's heavier regularization and simpler linear-plus-one-hot structure appears to generalize slightly better at this scale. This is a legitimate, useful finding for a course project — it is exactly the kind of result walk-forward validation exists to surface, rather than a result we should talk ourselves out of.

## Visualizing LightGBM's predictions on held-out stores

Numbers in a table are useful for ranking models, but a sanity-check plot catches failure modes that summary statistics can hide — for example a model that is right on average but systematically wrong on promo days. The cells below take the **last fold's** validation slice (the most recent, most representative 42-day window) and one store per `StoreType` (the first three store types encountered in the data), and plot actual vs. LightGBM-predicted Sales over that window. The comparison table itself is also written to `../reports/figures/model_comparison_table.csv` so it can be referenced from the final written report without re-running this notebook.

In [ ]:
Path("../reports/figures").mkdir(parents=True, exist_ok=True)
results_df.to_csv("../reports/figures/model_comparison_table.csv", index=False)

In [ ]:
import matplotlib.pyplot as plt
plot_df = last_fold_val_df.copy()
plot_df["lgbm_pred"] = last_fold_lgbm_pred
sample_stores = plot_df.drop_duplicates("Store").groupby("StoreType").first()["Store"].tolist()[:3]
fig, axes = plt.subplots(len(sample_stores), 1, figsize=(10, 3 * len(sample_stores)), sharex=True)
if len(sample_stores) == 1:
    axes = [axes]
for ax, s in zip(axes, sample_stores):
    sdf = plot_df[plot_df["Store"] == s].sort_values("Date")
    ax.plot(sdf["Date"], sdf["Sales"], label="Actual")
    ax.plot(sdf["Date"], sdf["lgbm_pred"], label="LightGBM Predicted")
    ax.set_title(f"Store {s} ({sdf['StoreType'].iloc[0]})")
    ax.legend()
fig.tight_layout()
from src.viz import save_fig
save_fig(fig, "../reports/figures/predicted_vs_actual_sample_stores.png")

## Refitting LightGBM on all trainable history for notebook 08

Walk-forward validation above exists to *choose and trust* a model, not to produce the model that eventually gets deployed — every fold deliberately withheld its last 42 days from training so it could be scored fairly. Once that evaluation is done, the standard next step is to refit the chosen model architecture on every trainable row (no held-out slice this time), so the deployed model has access to the most recent ~7 months of history that the last fold's model never saw.

This pipeline persists a `LGBMRegressor` here (`../models/lightgbm_global.pkl`) together with the exact feature-column contract it was trained on (`../models/feature_config.pkl`), for notebook 08 to load and generate the final forecast. Given the comparison above, this is worth flagging honestly: on this sampled dataset, Ridge was the stronger validated model on every fold, so a production-quality version of this pipeline would be well justified in persisting the Ridge pipeline instead of (or alongside) LightGBM. We still fit and save the LightGBM model here because it is the model this pipeline's downstream contract (notebook 08) expects — but the walk-forward numbers above, not this choice of what to serialize, are the honest record of which model actually validated better on this data.

In [ ]:
all_cat = set_categorical_dtypes(df, CATEGORICAL_FEATURE_COLS)
final_model = build_lightgbm_model()
final_model.fit(all_cat[FEATURE_COLS], all_cat["log_sales"])
Path("../models").mkdir(parents=True, exist_ok=True)
joblib.dump(final_model, "../models/lightgbm_global.pkl")
joblib.dump(
    {"numeric_cols": NUMERIC_FEATURE_COLS, "categorical_cols": CATEGORICAL_FEATURE_COLS, "feature_cols": FEATURE_COLS},
    "../models/feature_config.pkl",
)
print("Final global LightGBM model fit on all trainable rows and persisted for notebook 08.")

## Acceptance bar: did LightGBM clear it?

For this notebook, "success" is defined explicitly as: **LightGBM beats both baselines *and* Ridge on WMAPE in at least 4 of the 5 folds, with a mean MASE below 1.0.**

The cell below checks both halves of that bar against whatever `results_df` actually contains for *this* run, rather than asserting a fixed conclusion in prose -- re-running this notebook on a different dataset (e.g. the real Kaggle data instead of this synthetic smoke-test run) will re-evaluate the verdict instead of repeating stale numbers.

In [ ]:
wmape_cols_all = ["seasonal_naive_wmape", "moving_average_wmape", "ridge_wmape", "lightgbm_wmape"]
lightgbm_wins = (results_df["lightgbm_wmape"] == results_df[wmape_cols_all].min(axis=1)).sum()
mean_mase = results_df["lightgbm_mase"].mean()
n_folds_total = len(results_df)

mase_ok = mean_mase < 1.0
wmape_ok = lightgbm_wins >= 4

print(f"Mean LightGBM MASE: {mean_mase:.3f} -> {'meets' if mase_ok else 'does NOT meet'} the < 1.0 bar")
print(f"LightGBM had the lowest WMAPE in {lightgbm_wins}/{n_folds_total} folds -> "
      f"{'meets' if wmape_ok else 'does NOT meet'} the >=4/{n_folds_total} bar")

if mase_ok and wmape_ok:
    print("\nOverall: the acceptance bar IS met on this run.")
else:
    best_wmape_model = results_df[wmape_cols_all].mean().idxmin().replace("_wmape", "")
    print(
        f"\nOverall: the acceptance bar is NOT met on this run. Mean WMAPE is lowest for "
        f"'{best_wmape_model}', not lightgbm -- that is the better-validated model on this "
        f"particular run, even though the persisted artifact above is still the LightGBM "
        f"model notebook 08 is wired to consume. Re-running this notebook end-to-end on a "
        f"different dataset will recompute this verdict rather than repeat these numbers."
    )